# Pratica — Lezione 2: Dati e dataset in PyTorch

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife.

In questa pratica applicherai tu stesso quello che hai visto in classe: rappresentare i dati come un `Dataset`, servirli con un `DataLoader` y trabajar con un dataset de referencia de `torchvision`. Hay dos ejercicios:

1. Crear un `Dataset` personalizado sencillo y envolverlo en un `DataLoader`
2. Cargar el conjunto de test de MNIST y calcular cuántos lotes resultan para distintos tamaños de lote

Cada ejercicio tiene una celda de código con el andamiaje ya preparado y partes marcadas con `# TODO` para que completes tú. Intenta resolverlas antes de mirar la solución: aprenderás mucho más equivocándote y corrigiendo que leyendo directamente la respuesta.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

print("Versión de torch:", torch.__version__)

## Esercizio 1 — Un Dataset personalizzato

En la clase viste que un `Dataset` solo necesita responder a dos preguntas: cuántos elementos tiene (`__len__`) y cómo es el ejemplo número *i* (`__getitem__`). Vas a crear un `Dataset` muy simple a partir de una lista de números: cada elemento del dataset será ese valor original y su cuadrado.

Pistas:

- Hereda de `torch.utils.data.Dataset`
- En `__init__`, guarda la lista de números que recibas
- En `__len__`, devuelve cuántos números hay en la lista
- En `__getitem__`, devuelve una tupla `(numero, numero ** 2)` para la posición `idx`. Conviene devolver ambos valores como tensores (`torch.tensor(...)`), tal y como esperan el resto de utilidades de PyTorch
- Una vez definida la clase, crea una instancia con los números del uno al 20 y envuélvela en un `DataLoader` con `batch_size=4`
- Itera el `DataLoader` con un bucle `for` e imprime cada lote

In [ ]:
class DatasetDeCuadrados(Dataset):
    def __init__(self, numeros):
        # TODO: guarda la lista de numeros que recibe el constructor
        pass

    def __len__(self):
        # TODO: devuelve cuántos elementos tiene el dataset
        pass

    def __getitem__(self, idx):
        # TODO: devuelve la tupla (numero, numero al cuadrado) como tensores,
        # para la posición idx de la lista guardada en __init__
        pass


# TODO: crea una instancia de DatasetDeCuadrados con los números del 1 al 20 (usa range)
dataset_cuadrados = None

# TODO: envuelve dataset_cuadrados en un DataLoader con batch_size=4
loader_cuadrados = None

# TODO: itera loader_cuadrados con un bucle for e imprime cada lote

<details>
<summary>💡 Vedi soluzione</summary>

```python
class DatasetDeCuadrados(Dataset):
    def __init__(self, numeros):
        # Guardamos la lista de números que recibe el constructor
        self.numeros = list(numeros)

    def __len__(self):
        # El dataset tiene tantos elementos como números guardados
        return len(self.numeros)

    def __getitem__(self, idx):
        # Cada elemento es una tupla (número, su cuadrado), como tensores
        numero = self.numeros[idx]
        return torch.tensor(numero), torch.tensor(numero ** 2)


# Instancia del dataset con los números del 1 al 20
dataset_cuadrados = DatasetDeCuadrados(range(1, 21))

# DataLoader que sirve el dataset en lotes de 4 elementos
loader_cuadrados = DataLoader(dataset_cuadrados, batch_size=4)

# Iteramos sobre los lotes
for numeros, cuadrados in loader_cuadrados:
    print("Números:", numeros, "-> Cuadrados:", cuadrados)
```
</details>

## Esercizio 2 — DataLoader sul test di MNIST

Ahora vas a trabajar con el conjunto de test de MNIST, el mismo que viste en la clase, pero fijándote en cuántos lotes resultan según el `batch_size` que elijas.

Pistas:

- Carga `datasets.MNIST` con `train=False` y `download=True`, aplicando `transforms.ToTensor()` como transformación
- Crea un `DataLoader` sobre ese dataset con `batch_size=32`
- El número de lotes de un `DataLoader` es simplemente `len(loader)`: PyTorch ya calcula por ti cuántas porciones completas (y, si sobran ejemplos, una final más pequeña) resultan de dividir el dataset entre el `batch_size`
- Comprueba a mano que el resultado tiene sentido: si el dataset de test de MNIST tiene 10.000 ejemplos y usas un `batch_size` de 32, ¿cuántos lotes esperarías, redondeando hacia arriba?
- Repite el cálculo con `batch_size=64` y compara

In [ ]:
# TODO: carga el dataset de test de MNIST (train=False, download=True)
# aplicando transforms.ToTensor() como transformación
testset = None

# TODO: crea un DataLoader sobre testset con batch_size=32
loader_32 = None

# TODO: crea un segundo DataLoader sobre testset con batch_size=64
loader_64 = None

# TODO: imprime cuántos ejemplos tiene testset y cuántos lotes resultan
# en cada DataLoader (usa len(...) en los tres casos)

<details>
<summary>💡 Vedi soluzione</summary>

```python
transformacion = transforms.ToTensor()

# Conjunto de test de MNIST
testset = datasets.MNIST(
    root="./datos",
    train=False,
    download=True,
    transform=transformacion,
)

# Dos DataLoaders sobre el mismo dataset, con distinto tamaño de lote
loader_32 = DataLoader(testset, batch_size=32)
loader_64 = DataLoader(testset, batch_size=64)

print("Ejemplos en testset:", len(testset))
print("Lotes con batch_size=32:", len(loader_32))
print("Lotes con batch_size=64:", len(loader_64))
```

Con 10.000 esempi, una dimensione di lotto di 32 produce 313 lotti (312 lotti completi di 32 e un ultimo lotto di 16 esempi), e una dimensione di lotto di 64 produce 157 lotti (156 completi e uno finale di 16). In entrambi i casi, `DataLoader` calcola il numero di lotti arrotondando verso l'alto il risultato della divisione del totale di esempi per la dimensione del lotto.
</details>

## Per continuare a esercitarti

Se ti è rimasto tempo, provare a modificare `DatasetDeCuadrados` dell'esercizio 1 perché restituisca il cubo del numero invece del quadrato, oppure cambia `batch_size` nell'esercizio 2 con un valore che non sia divisore esatto di 10.000 (ad esempio, 47) e conferma che il numero di lotti abbia ancora senso. È il modo migliore per verificare che capisci come `Dataset` e `DataLoader` lavorano insieme, e non solo che sai copiare lo schema.